In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7
)

In [3]:
class JokeState(TypedDict):

  topic:str
  joke:str
  explanation:str

In [4]:
def generate_joke(state:JokeState):

    prompt = f"Generate a joke on the topic {state['topic']}"
    response = llm.invoke(prompt).content

    return {'joke':response}

In [5]:
def generate_explanation(state:JokeState):

  prompt = f"Write an explanation for the joke - {state['joke']}"

  response = llm.invoke(prompt).content

  return {'explanation':response}

In [6]:
graph = StateGraph(JokeState)

# Add nodes
graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer= checkpointer)

In [7]:
config1 = {"configurable":{"thread_id":1}}
workflow.invoke({'topic':'pizza'},config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?\n\nBecause it needed a little *dough* and wanted to *rise* to the occasion! 🍕😄',
 'explanation': '**Explanation of the joke**\n\n> **Why did the pizza apply for a job?**  \n> **Because it needed a little *dough* and wanted to *rise* to the occasion!** 🍕😄  \n\nThe humor comes from two classic English word‑plays (puns) that tie together the world of pizza and the world of work.\n\n| Phrase in the punchline | What it means in everyday language | What it means in the pizza‑related sense |\n|-------------------------|-----------------------------------|------------------------------------------|\n| **“dough”** | Slang for money (e.g., “I need more dough”). | The literal dough that forms the base of a pizza. |\n| **“rise”** | To improve, advance, or meet a challenge (“rise to the occasion”). | The way pizza dough (and the pizza itself) expands and puffs up during fermentation or baking, thanks to yeast. |\n\nSo the joke works be

In [8]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it needed a little *dough* and wanted to *rise* to the occasion! 🍕😄', 'explanation': '**Explanation of the joke**\n\n> **Why did the pizza apply for a job?**  \n> **Because it needed a little *dough* and wanted to *rise* to the occasion!** 🍕😄  \n\nThe humor comes from two classic English word‑plays (puns) that tie together the world of pizza and the world of work.\n\n| Phrase in the punchline | What it means in everyday language | What it means in the pizza‑related sense |\n|-------------------------|-----------------------------------|------------------------------------------|\n| **“dough”** | Slang for money (e.g., “I need more dough”). | The literal dough that forms the base of a pizza. |\n| **“rise”** | To improve, advance, or meet a challenge (“rise to the occasion”). | The way pizza dough (and the pizza itself) expands and puffs up during fermentation or baking, thanks to yeast. |\n\n

In [9]:
config2 = {"configurable":{"thread_id":"2"}}
workflow.invoke({'topic':'pasta'},config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti go to therapy?  \n\nBecause it couldn’t *knot* handle all the tangled emotions—it was always getting *pasta* its limits! 🍝😄',
 'explanation': '**Joke:**  \n*Why did the spaghetti go to therapy?*  \n*Because it couldn’t **knot** handle all the tangled emotions—it was always getting **pasta** its limits! 🍝😄*\n\n---\n\n### Why it’s funny – a step‑by‑step breakdown\n\n| Part of the joke | What it says literally | The wordplay / pun | Why it makes you laugh |\n|------------------|------------------------|-------------------|-----------------------|\n| **“spaghetti”** | A long, thin pasta that’s usually served in a bowl. | The joke treats spaghetti as if it were a person with feelings. Anthropomorphizing food (giving it human traits) is inherently goofy. | The absurd image of a noodle needing therapy is already a funny mental picture. |\n| **“couldn’t **knot** handle”** | Uses the verb *knot* (to tie a knot) instead of *not*. | *Knot* sounds

### Time travel

In [12]:
workflow.get_state({'configurable':{"thread_id":"1","checkpoint_id":"1f1d411c-b1be-66f5-8000-94e6c233240d"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1d411c-b1be-66f5-8000-94e6c233240d'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-10-30T03:27:11.233178+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d411c-b1b9-6a24-bfff-61c079f59ecf'}}, tasks=(PregelTask(id='745011d3-37cd-1558-6112-e6a78978383a', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza apply for a job?\n\nBecause it needed a little *dough* and wanted to *rise* to the occasion! 🍕😄'}),), interrupts=())

In [13]:
workflow.invoke(None,{'configurable':{"thread_id":"1","checkpoint_id":"1f1d411c-b1be-66f5-8000-94e6c233240d"}})

{'topic': 'pizza',
 'joke': 'Why did the slice of pizza go to therapy?  \n\nBecause it couldn’t stop feeling *crust*y and was always getting into cheesy arguments! 🍕😄',
 'explanation': '**What the joke is saying**\n\n> *Why did the slice of pizza go to therapy?*  \n> *Because it couldn’t stop feeling **crust*y and was always getting into **cheesy** arguments! 🍕😄*\n\nThe humor comes from a mix of three things:\n\n| Element | Why it’s funny | Explanation |\n|---------|----------------|-------------|\n| **“Crust*y”** | A play on the word *crusty* (meaning irritable or “hard‑headed”) and the literal **crust** of a pizza. | A pizza slice has a crust, so saying it feels “crusty” turns a personality trait into a physical pizza feature. |\n| **“Cheesy arguments”** | A double‑meaning pun on *cheesy* (slang for corny or over‑the‑top) and the cheese that tops most pizzas. | “Cheesy arguments” sounds like silly, over‑done fights, but in the pizza world it literally involves cheese. |\n| **Therapy*

In [14]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the slice of pizza go to therapy?  \n\nBecause it couldn’t stop feeling *crust*y and was always getting into cheesy arguments! 🍕😄', 'explanation': '**What the joke is saying**\n\n> *Why did the slice of pizza go to therapy?*  \n> *Because it couldn’t stop feeling **crust*y and was always getting into **cheesy** arguments! 🍕😄*\n\nThe humor comes from a mix of three things:\n\n| Element | Why it’s funny | Explanation |\n|---------|----------------|-------------|\n| **“Crust*y”** | A play on the word *crusty* (meaning irritable or “hard‑headed”) and the literal **crust** of a pizza. | A pizza slice has a crust, so saying it feels “crusty” turns a personality trait into a physical pizza feature. |\n| **“Cheesy arguments”** | A double‑meaning pun on *cheesy* (slang for corny or over‑the‑top) and the cheese that tops most pizzas. | “Cheesy arguments” sounds like silly, over‑done fights, but in the pizza world it literally involves che